UNIFESP X-Ray Body Part Classification — Step 2 (baseline models)

Runs the 8 baseline backbones (VGG16, InceptionV3, ResNet50, DenseNet121, EfficientNetB3, EfficientNetV2-S, ConvNeXt-Tiny, Swin-Tiny), using both GPUs in parallel, with checkpoint/auto-resume so a 12-hour session cutoff never loses progress — just re-run the same cell in a new session.

Before running: attach both datasets in Notebook Settings → Data, and turn on GPU T4 x2 in Notebook Settings → Accelerator.

1. Environment setup (auto-detects both attached datasets)

No hardcoded folder names -- this walks every dataset mounted under /kaggle/input and finds:

the pipeline dataset: the one that contains a src/ folder somewhere in it
the images dataset: the one that contains both a train and a test folder

This avoids the earlier error (FileNotFoundError: .../felipekitamura/src), which happened because the pipeline zip hadn't actually been uploaded into the dataset yet -- if you still see "pipeline dataset not found" below, it means the dataset on Kaggle is still missing src//index.csv/splits/; re-check Kaggle "New Version" uploaded unifesp_pipeline.zip and that the dataset is attached to this notebook (Notebook Settings -> Data).

In [ ]:
import os

# Full, unlimited-depth walk of /kaggle/input -- deliberately not assuming any
# particular nesting (plain /kaggle/input/<slug>/... vs the deeper
# /kaggle/input/datasets/<owner>/<slug>/... some Kaggle environments use).
# This only stats directory names, so walking the whole tree is cheap even
# though the images dataset has ~2500 files.
pipeline_root = None
images_root = None

print("Tree under /kaggle/input (dirs only, depth<=2, for debugging -- capped at 200 lines):")
_printed = 0
for cur, dirs, files in os.walk("/kaggle/input"):
    depth = cur.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 2 and _printed < 200:
        print("  " * depth + os.path.basename(cur) + "/")
        _printed += 1
    elif _printed == 200:
        print("  ... (truncated, too many entries -- this is fine, detection below doesn't depend on this print)")
        _printed += 1

    if "src" in dirs and pipeline_root is None:
        pipeline_root = os.path.join(cur, "src")
    if "train" in dirs and "test" in dirs and images_root is None:
        images_root = cur
        dirs[:] = [d for d in dirs if d not in ("train", "test")]  # found it, stop descending into the ~2500 image files

print()
print("Detected pipeline root (parent of src/):", pipeline_root)
print("Detected images root (has train/ and test/):", images_root)

assert pipeline_root is not None, (
    "Could not find a 'src' folder anywhere under /kaggle/input (see the "
    "tree printed above). Make sure unifesp_pipeline.zip was uploaded as a "
    "NEW VERSION of your Kaggle dataset (not just KAGGLE_SETUP.md), and "
    "that dataset is attached to this notebook (Notebook Settings -> Data)."
)
assert images_root is not None, (
    "Could not find a folder with both train/ and test/ subfolders anywhere "
    "under /kaggle/input (see the tree printed above). Attach the original "
    "felipekitamura/unifesp-xray-bodypart-classification dataset too "
    "(Notebook Settings -> Data -> Add Input)."
)
PIPELINE_DATASET = os.path.dirname(pipeline_root)
IMAGES_DATASET = images_root

Tree under /kaggle/input (dirs only, depth<=2, for debugging -- capped at 200 lines):
input/
  datasets/
    felipekitamura/
    noorabdullah02/

Detected pipeline root (parent of src/): /kaggle/input/datasets/noorabdullah02/felipekitamura/src
Detected images root (has train/ and test/): /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification


In [ ]:
import shutil, sys

shutil.copytree(os.path.join(PIPELINE_DATASET, "src"), "/kaggle/working/src", dirs_exist_ok=True)
sys.path.insert(0, "/kaggle/working")

os.environ["UNIFESP_DATA_ROOT"] = IMAGES_DATASET
os.environ["UNIFESP_INDEX_ROOT"] = PIPELINE_DATASET
os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"
os.environ["UNIFESP_CACHE_DIR"] = "/kaggle/working/image_cache"

print("DATA_ROOT  ->", os.environ["UNIFESP_DATA_ROOT"])
print("INDEX_ROOT ->", os.environ["UNIFESP_INDEX_ROOT"])
print("index.csv exists:", os.path.exists(os.path.join(PIPELINE_DATASET, "index.csv")))
print("splits/folds.csv exists:", os.path.exists(os.path.join(PIPELINE_DATASET, "splits", "folds.csv")))

DATA_ROOT  -> /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification
INDEX_ROOT -> /kaggle/input/datasets/noorabdullah02/felipekitamura
index.csv exists: True
splits/folds.csv exists: True


In [ ]:
!pip install -q timm albumentations iterative-stratification thop grad-cam

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# sanity check: imports work, index/folds load, GPU count visible
import torch
from src import config as C
import pandas as pd

print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("INDEX_CSV:", C.INDEX_CSV, "exists:", os.path.exists(C.INDEX_CSV))
print("FOLDS_CSV:", C.FOLDS_CSV, "exists:", os.path.exists(C.FOLDS_CSV))

idx = pd.read_csv(C.INDEX_CSV)
print(idx['split'].value_counts())

CUDA available: True | GPU count: 2
INDEX_CSV: /kaggle/input/datasets/noorabdullah02/felipekitamura/index.csv exists: True
FOLDS_CSV: /kaggle/input/datasets/noorabdullah02/felipekitamura/splits/folds.csv exists: True
split
train    1738
test      743
Name: count, dtype: int64


2. Architecture stats (params, FLOPs, inference time)

Cheap — no training, just builds each model once and measures it.

In [ ]:
!cd /kaggle/working && python -m src.compare_backbones

[vgg16] building...
  params=134.35M  flops=20.522840064G  inference=14.60ms
[inception_v3] building...
  params=21.83M  flops=3.847670112G  inference=12.20ms
[resnet50] building...
  params=23.55M  flops=5.396543488G  inference=5.53ms
[densenet121] building...
  params=6.98M  flops=3.700743168G  inference=15.56ms
[efficientnet_b3] building...
  params=10.73M  flops=1.254972992G  inference=12.44ms
[efficientnetv2_s] building...
  params=20.21M  flops=3.538001664G  inference=16.80ms
[convnext_tiny] building...
  params=27.84M  flops=5.81853312G  inference=8.04ms
[swin_tiny] building...
  params=27.54M  flops=4.371100704G  inference=10.87ms

Saved: /kaggle/working/runs/backbone_architecture_stats.csv
        backbone  img_size   params_M   flops_G  inference_ms_per_image
           vgg16       256 134.350678 20.522840               14.595222
    inception_v3       256  21.830646  3.847670               12.201325
        resnet50       256  23.553110  5.396543                5.528498
    

2b. (Optional, recommended) Pre-cache DICOM -> PNG

UnifespDataset caches lazily on its own, but pre-warming here avoids BOTH backbones in each GPU pair redundantly decoding the same ~1700 training DICOMs during their first epoch. Top finishers in the real UNIFESP Kaggle competition reported DICOM decoding as their biggest bottleneck -- this takes a couple of minutes once and speeds up every run after it.

In [ ]:
!cd /kaggle/working && python -m src.cache_images

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
Found 2481 files. Caching to /kaggle/working/image_cache ...
  250/2481 (22.5s elapsed)
  500/2481 (45.8s elapsed)
  750/2481 (71.7s elapsed)
  1000/2481 (94.5s elapsed)
  1250/2481 (116.8s elapsed)
  1500/2481 (139.7s elapsed)
  1750/2481 (167.1s elapsed)
  2000/2481 (189.5s elapsed)
  2250/2481 (211.7s elapsed)
Done. Cached 2481 images in 233.5s -> /kaggle/working/image_cache


3. Train all 8 baselines on fold 0, two at a time (one per GPU)

--max_hours 5 per run leaves plenty of room inside the 12h session for a pair to finish. If a session still gets cut off mid-run, just re-run this same cell later (new session, same datasets attached) — training resumes from each backbone's checkpoint_last.pt instead of restarting.

In [ ]:
import subprocess

backbones = ["vgg16", "inception_v3", "resnet50", "densenet121",
             "efficientnet_b3", "efficientnetv2_s", "convnext_tiny", "swin_tiny"]

def launch(backbone, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"/kaggle/working/{backbone}.log", "w")
    return subprocess.Popen(
        ["python", "-m", "src.train", "--backbone", backbone, "--fold", "0",
         "--epochs", "30", "--batch_size", "32", "--max_hours", "5"],
        cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT,
    )

failed = []
for i in range(0, len(backbones), 2):
    pair = backbones[i:i+2]
    print("launching:", pair)
    procs = list(zip(pair, [launch(b, gpu) for gpu, b in enumerate(pair)]))
    for name, p in procs:
        rc = p.wait()
        if rc != 0:
            failed.append(name)
            print(f"  [FAILED] {name} exited with code {rc} -- check /kaggle/working/{name}.log")
        else:
            print(f"  [ok] {name}")
    print("finished pair:", pair)

if failed:
    print("\nBackbones that failed:", failed, "-- check their .log files before trusting the comparison table.")
else:
    print("\nAll 8 backbones finished successfully.")

launching: ['vgg16', 'inception_v3']
  [ok] vgg16
  [ok] inception_v3
finished pair: ['vgg16', 'inception_v3']
launching: ['resnet50', 'densenet121']
  [ok] resnet50
  [ok] densenet121
finished pair: ['resnet50', 'densenet121']
launching: ['efficientnet_b3', 'efficientnetv2_s']
  [ok] efficientnet_b3
  [ok] efficientnetv2_s
finished pair: ['efficientnet_b3', 'efficientnetv2_s']
launching: ['convnext_tiny', 'swin_tiny']
  [ok] convnext_tiny
  [ok] swin_tiny
finished pair: ['convnext_tiny', 'swin_tiny']

All 8 backbones finished successfully.


Check progress of a specific backbone any time (run in a new cell while training is in progress):

In [ ]:
!tail -n 30 /kaggle/working/efficientnet_b3.log

epoch 6: train_loss=0.0258 val_loss=0.1916 macro_f1=0.8081 (11.5s)
epoch 7: train_loss=0.0185 val_loss=0.2053 macro_f1=0.8395 (11.7s)
  -> new best macro_f1=0.8395, saved checkpoint_best.pt
epoch 8: train_loss=0.0196 val_loss=0.2019 macro_f1=0.8254 (11.7s)
epoch 9: train_loss=0.0143 val_loss=0.2066 macro_f1=0.8422 (11.5s)
  -> new best macro_f1=0.8422, saved checkpoint_best.pt
epoch 10: train_loss=0.0108 val_loss=0.2119 macro_f1=0.8470 (11.4s)
  -> new best macro_f1=0.8470, saved checkpoint_best.pt
epoch 11: train_loss=0.0132 val_loss=0.2055 macro_f1=0.8431 (11.6s)
epoch 12: train_loss=0.0086 val_loss=0.2361 macro_f1=0.8446 (12.1s)
epoch 13: train_loss=0.0095 val_loss=0.2363 macro_f1=0.8636 (11.9s)
  -> new best macro_f1=0.8636, saved checkpoint_best.pt
epoch 14: train_loss=0.0096 val_loss=0.2402 macro_f1=0.8412 (12.0s)
epoch 15: train_loss=0.0062 val_loss=0.2465 macro_f1=0.8476 (11.9s)
epoch 16: train_loss=0.0048 val_loss=0.2591 macro_f1=0.8511 (12.1s)
epoch 17: train_loss=0.0036 val_

4. Final comparison table

Merges architecture stats with trained performance (best macro-F1/AUC/kappa, avg epoch training time) into one CSV.

In [ ]:
!cd /kaggle/working && python -m src.aggregate_results --fold 0

Saved: /kaggle/working/runs/comparison_table_fold0.csv
        backbone  img_size   params_M   flops_G  inference_ms_per_image  epochs_trained  epochs_to_best  avg_epoch_train_s  best_macro_f1  best_micro_f1  best_macro_auc  best_kappa_flat  best_subset_accuracy
           vgg16       256 134.350678 20.522840               14.595222              30              19          19.951027       0.828417       0.891331        0.991507         0.885491              0.829670
    inception_v3       256  21.830646  3.847670               12.201325              30              19           9.627404       0.862120       0.915935        0.989408         0.911536              0.865385
        resnet50       256  23.553110  5.396543                5.528498              30              28          10.074994       0.829322       0.892382        0.992379         0.886546              0.832418
     densenet121       256   6.976406  3.700743               15.563361              30              25          

4b. Leaderboard-comparable score (sample-mean F1)

The real UNIFESP Kaggle competition (kaggle.com/competitions/unifesp-x-ray- body-part-classifier) scored submissions with Mean F1-Score: one F1 per image between predicted/true label sets, averaged over images -- not the same as macro_f1 (averaged per class) that Step 2's comparison table uses. This recomputes that exact metric from the checkpoints we already trained (no retraining needed) so we can see where we actually stand against the real leaderboard (1st place: 0.934, private LB, 115 teams).

In [ ]:
!cd /kaggle/working && python -m src.evaluate_checkpoints --fold 0

indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
[vgg16] sample_mean_f1=0.9071  (macro_f1=0.8284, epoch 18)
[inception_v3] sample_mean_f1=0.9169  (macro_f1=0.8621, epoch 18)
[resnet50] sample_mean_f1=0.9120  (macro_f1=0.8293, epoch 27)
[densenet121] sample_mean_f1=0.9266  (macro_f1=0.8736, epoch 24)
[efficientnet_b3] sample_mean_f1=0.9254  (macro_f1=0.8673, epoch 25)
[efficientnetv2_s] sample_mean_f1=0.9231  (macro_f1=0.8967, epoch 12)
[convnext_tiny] sample_mean_f1=0.8995  (macro_f1=0.8474, epoch 27)
[swin_tiny] sample_mean_f1=0.9272  (macro_f1=0.8888, epoch 21)

Saved: /kaggle/working/runs/leaderboard_comparable_fold0.csv
        backbone  sample_mean_f1  macro_f1  micro_f1  subset_accuracy
       swin_tiny        0.927198  0.888820  0.919054         0.873626
     densenet121        0.926557  0.873613  0.921348         0.873626
 efficientnet_b3        0.925366  0.867259  0.924051         0.884615
efficientnetv2_s        0.92307

In [ ]:
import pandas as pd
pd.read_csv("/kaggle/working/runs/comparison_table_fold0.csv")

!tail -n 50 /kaggle/working/vgg16.log

/kaggle/working/src/engine.py:153: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if device.type == "cuda" else None
train: 1374 images | val: 364 images
indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
found 2481 .dcm files
epoch 0: train_loss=0.6903 val_loss=0.5153 macro_f1=0.3583 (19.7s)
  -> new best macro_f1=0.3583, saved checkpoint_best.pt
epoch 1: train_loss=0.5150 val_loss=0.3326 macro_f1=0.4534 (27.5s)
  -> new best macro_f1=0.4534, saved checkpoint_best.pt
epoch 2: train_loss=0.3189 val_loss=0.3321 macro_f1=0.5328 (18.9s)
  -> new best macro_f1=0.5328, saved checkpoint_best.pt
epoch 3: train_loss=0.3097 val_loss=0.3653 macro_f1=0.4792 (19.8s)
epoch 4: train_loss=0.2412 val_loss=0.2864 macro_f1=0.5135 (21.5s)
epoch 5: train_loss=0.1851 val_loss=0.2041 macro_f1=0.6374 (19.7s)
  -> new best macro_f1=0.6374

5. Before this session ends

/kaggle/working is wiped when the session closes unless you save it. Click "Save Version" now, or at minimum download the runs/ folder and the two CSVs above, so checkpoints and results survive into your next session.